In [1]:
import pandas as pd

In [9]:
import pandas as pd

df = pd.read_excel("Telco-Customer-Churn.csv.xlsx")

print(df.shape)
print(df.head())


(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

In [10]:
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
Total Charges       0
Churn               0
dtype: int64


In [12]:
print(df.columns)

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'Total Charges', 'Churn'],
      dtype='object')


In [14]:
df_cleaned = df.dropna(subset=['Total Charges'])

df_cleaned['Total Charges'] = df_cleaned['Total Charges'].fillna(df_cleaned['Total Charges'].median())


In [16]:
df_cleaned['tenure'] = df_cleaned['tenure'].ffill()

In [17]:
df_cleaned['MonthlyCharges'] = df_cleaned['MonthlyCharges'].fillna(df_cleaned['MonthlyCharges'].median())

In [18]:
df_cleaned.to_csv("Telco_Cleaned.csv", index=False)

In [19]:
print("Cleaning complete! Cleaned dataset saved as Telco_Cleaned.csv")

Cleaning complete! Cleaned dataset saved as Telco_Cleaned.csv


In [21]:
!pip install mysql-connector-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 56.7 MB/s eta 0:00:00


In [22]:
import mysql.connector


In [28]:
import sqlite3
import pandas as pd

# Connect to SQLite (creates a local DB file)
conn = sqlite3.connect("telecom.db")

# ✅ Step 1: Add flag columns in Pandas before saving
df_cleaned.columns = df_cleaned.columns.str.replace(" ", "")
df_cleaned["TotalCharges_flag"] = None
df_cleaned["MonthlyCharges_flag"] = None

# ✅ Step 2: Save DataFrame into SQL
df_cleaned.to_sql("telco_customers", conn, if_exists="replace", index=False)

# ✅ Step 3: Run SQL updates
conn.execute("""
UPDATE telco_customers
SET TotalCharges_flag = CASE
    WHEN TotalCharges IS NULL THEN 'Imputed'
    ELSE 'Original'
END
WHERE customerID IS NOT NULL;
""")

conn.execute("""
UPDATE telco_customers
SET MonthlyCharges_flag = CASE
    WHEN MonthlyCharges IS NULL THEN 'Imputed'
    ELSE 'Original'
END
WHERE customerID IS NOT NULL;
""")

conn.commit()

# ✅ Step 4: Export back to CSV
df_sql = pd.read_sql("SELECT * FROM telco_customers", conn)
df_sql.to_csv("Telco_Final.csv", index=False)

print("Export complete! File saved as Telco_Final.csv")


Export complete! File saved as Telco_Final.csv


In [29]:
df_check = pd.read_csv("Telco_Final.csv")
df_check.head()


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalCharges_flag,MonthlyCharges_flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,Original,Original
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.50,No,Original,Original
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,Original,Original
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,Original,Original
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,Original,Original


In [30]:
from google.colab import drive
drive.mount('/content/drive')

# Save into your Drive
df_check.to_csv("/content/drive/MyDrive/Telco_Final.csv", index=False)


Mounted at /content/drive


In [31]:
df_sql.to_csv("/content/drive/MyDrive/Telco_Final.csv", index=False)

In [32]:
import os

# Rename the file in Google Drive
old_path = "/content/drive/MyDrive/Telco_Final.csv"
new_path = "/content/drive/MyDrive/Customer_Data_Cleaning_Pipeline.csv"

os.rename(old_path, new_path)

print("File renamed successfully!")


File renamed successfully!
